# VTL Kernel Metrics: Compositional Analysis for AI-Generated Images

## What This Notebook Does

## Companion to: "A Generative Field Framework for Measuring Spatial Priors in Image Models"

Collapse ≠ failure. The center is the delta of choice.

This notebook implements **kernel-based compositional metrics** that measure spatial geometry in images—dimensions that standard evaluation metrics (FID, IS, KID, CLIPScore, T2I-CompBench, GenEval) cannot detect.

While existing metrics measure:
- ✅ Semantic correctness (does it match the text?)
- ✅ Feature-space realism (does it look plausible?)

They are completely blind to:
- ❌ Placement bias (central vs. off-center)
- ❌ Void logic (negative space handling)
- ❌ Packing density (material compression)
- ❌ Cohesion (structural integrity)
- ❌ Radial collapse (default spatial priors)

This notebook closes that gap by directly measuring compositional geometry.

---

## The Five Kernel Primitives

This system measures images using five geometric primitives:

| Metric | Symbol | What It Measures | Interpretation |
|--------|--------|------------------|----------------|
| **Placement Offset** | Δx | Distance of mass centroid from frame center | < 0.05 = centered (radial collapse risk)<br>0.05-0.15 = moderate offset<br>> 0.15 = strong asymmetry |
| **Void Ratio** | rᵥ | Proportion of empty/negative space | < 0.3 = void collapse<br>0.3-0.6 = balanced<br>> 0.6 = void-dominant |
| **Packing Density** | ρᵣ | Material compression within bounding box | < 0.3 = dispersed<br>0.3-0.7 = moderate<br>> 0.7 = tight clustering |
| **Cohesion** | μ | Structural stability (variance from centroid) | < 0.4 = fragmented<br>0.4-0.7 = moderate<br>> 0.7 = unified structure |
| **Peripheral Pull** | xₚ | Field invariant derived from above metrics | < 0.2 = central attractor dominant<br>0.2-0.4 = moderate<br>> 0.4 = anti-RCP behavior |

**Field Invariant (xₚ):** Composite measure indicating whether composition resists or succumbs to radial collapse prior (RCP)—the tendency of AI models to center subjects with radial lighting.

**μ** is implemented as dispersion around the centroid, approximating structural convexity rather than texture coherence.”

---

## How to Use This Notebook

### **Step 1: Setup (Run Once)**
Run these cells in order:
- **Cell 1:** Install dependencies
- **Cell 2:** Import libraries
- **Cell 4:** Define all metric functions

### **Step 2: Analyze a Single Image**

**Option A: Quick Single Analysis**
1. **Cell 3:** Upload one image
2. **Cell 5:** Calculate metrics using saliency detection (recommended)
3. **Cell 6:** Visualize results with centroid and bounding box overlay

**Option B: Compare All Detection Methods**
1. **Cell 3:** Upload one image
2. **Cell 8:** Run all three detection methods (saliency, edges, Otsu) side-by-side
   - See which method works best for your image
   - Check agreement analysis (low variance = methods agree)

### **Step 3: Batch Processing (Optional)**

**Cell 9:** Process multiple images at once
- Upload several images
- Get comparison table showing metrics across all images
- Identify patterns (e.g., "80% of Model A's outputs are centered")

---

## Detection Methods Explained

The notebook offers three subject-detection methods:

| Method | Best For | Limitations |
|--------|----------|-------------|
| **Saliency** (recommended) | Complex backgrounds, gradients, most images | Slower processing |
| **Edges** | Clean images with strong subject boundaries | Can miss subtle subjects |
| **Otsu** | High-contrast images with simple backgrounds | Fooled by gradient backgrounds |

**Default recommendation:** Use **saliency detection** unless you have specific reasons to use another method.

---

## Interpreting Results

### **Radial Collapse Prior (RCP) Detection**

Images with **Δx < 0.05** and **xₚ < 0.3** indicate **Radial Collapse Prior**:
- Subject locked to frame center
- Weak peripheral features
- Likely generated by model with strong central bias

### **Anti-RCP Behavior**

Images with **Δx > 0.15** or **xₚ > 0.4** show compositional freedom:
- Deliberate asymmetry
- Strong void discipline
- Model capable of off-center compositions

### **Quick Diagnostic Output**

The notebook provides instant assessment:
- 🔴 **RADIAL COLLAPSE DETECTED** - centered + weak peripheral pull
- ⚠️ **CENTERED BUT ANTI-RCP** - centered with strong peripheral features
- ✅ **STRONG ASYMMETRY** - off-center with good void discipline
- ✅ **MODERATE ASYMMETRY** - compositional offset present
- ⚠️ **BALANCED** - moderate across dimensions

---

## Example Use Cases

### **1. Model Comparison**
Compare two AI image generators:
- Upload outputs from both models
- Check mean Δx values
- Model with lower Δx has stronger central bias

### **2. Prompt Engineering Validation**
Test if anti-collapse prompts work:
- Generate images with standard vs. structural prompts
- Compare xₚ scores
- Higher xₚ = better compositional control

### **3. Dataset Analysis**
Evaluate training data composition:
- Batch process dataset
- Check distribution of Δx, rᵥ, ρᵣ
- Identify compositional biases in training set

### **4. Version Control**
Track model updates:
- Measure kernel metrics across model versions
- Detect spatial prior drift
- Flag unexpected compositional changes

---

## What This Notebook Demonstrates

This measurement infrastructure exists and works **right now**—unlike FID/CLIP/T2I-CompBench which require:
- Reference datasets
- Batch processing
- No single-image analysis capability

**Key advantages:**
- ✅ Works on individual images immediately
- ✅ No reference dataset required
- ✅ Interpretable geometric measurements
- ✅ Detects spatial priors invisible to existing metrics
- ✅ Model-agnostic (works on any image source)

---

## Citation

If you use these metrics in research or development:
```
Russell Parrish. Visual Thinking Lens: Kernel-Based Compositional Metrics, 2025.
ORCID: 0009-0008-9781-7995
```

---

## Need Help?

**Common issues:**
- **"No material detected"** → Try different detection method (Cell 8)
- **Background detected as subject** → Use saliency instead of Otsu
- **Methods strongly disagree** → Image has ambiguous subject boundaries

**For complex cases:** Run Cell 8 to compare all three methods and manually verify which gives most accurate subject detection.

---

**Ready to start?** Run Cell 1 → Cell 2 → Cell 4, then upload your first image in Cell 3! 🚀

In [ ]:
!pip uninstall -y opencv-python opencv-contrib-python
!pip install pillow numpy matplotlib scipy opencv-contrib-python

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from scipy.spatial import ConvexHull
from google.colab import files
import cv2

In [ ]:
# Upload your image
uploaded = files.upload()

# Get the filename
image_filename = list(uploaded.keys())[0]
print(f"Uploaded: {image_filename}")

# Load and display the image
img = Image.open(image_filename)
plt.figure(figsize=(10, 8))
plt.imshow(img)
plt.axis('off')
plt.title("Uploaded Image")
plt.show()

In [ ]:
def calculate_kernel_metrics(image_path, method='saliency'):
    """
    Calculate VTL Kernel metrics for an image.

    Parameters:
    - image_path: path to image file
    - method: 'saliency', 'edges', or 'otsu' (default: 'saliency')

    Returns:
    - Dictionary of kernel metrics
    """
    # Load image
    img = Image.open(image_path).convert('RGB')
    img_array = np.array(img)

    height, width = img_array.shape[:2]

    # Convert to grayscale
    gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)

    # Choose detection method
    if method == 'saliency':
        # Use saliency detection to find visually important regions
        saliency_detector = cv2.saliency.StaticSaliencySpectralResidual_create()
        success, saliency_map = saliency_detector.computeSaliency(img_array)

        if not success:
            print("Saliency detection failed, falling back to Otsu thresholding")
            method = 'otsu'
        else:
            # Convert saliency map to 8-bit
            saliency_map = (saliency_map * 255).astype(np.uint8)

            # Threshold the saliency map
            _, material_mask = cv2.threshold(saliency_map, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

            # Apply morphological operations to clean up the mask
            kernel = np.ones((5,5), np.uint8)
            material_mask = cv2.morphologyEx(material_mask, cv2.MORPH_CLOSE, kernel)
            material_mask = cv2.morphologyEx(material_mask, cv2.MORPH_OPEN, kernel)

    if method == 'edges':
        # Use Canny edge detection
        edges = cv2.Canny(gray, 50, 150)

        # Dilate edges to create regions
        kernel = np.ones((5,5), np.uint8)
        edges_dilated = cv2.dilate(edges, kernel, iterations=2)

        # Fill enclosed regions
        contours, _ = cv2.findContours(edges_dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        material_mask = np.zeros_like(gray)
        cv2.drawContours(material_mask, contours, -1, 255, thickness=cv2.FILLED)

    if method == 'otsu':
        # Use Otsu's automatic thresholding (original method)
        _, material_mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # Convert to boolean
    material_mask = material_mask > 0

    # Get material pixel coordinates
    material_coords = np.argwhere(material_mask)

    if len(material_coords) == 0:
        print(f"Warning: No material detected with method '{method}'.")
        return None

    # material_coords are in (row, col) = (y, x) format
    y_coords = material_coords[:, 0]
    x_coords = material_coords[:, 1]

    # Calculate metrics
    metrics = {}
    metrics['method'] = method

    # 1. Δx - Placement Offset
    cx = np.mean(x_coords)
    cy = np.mean(y_coords)
    center_x = width / 2
    center_y = height / 2

    delta_x = abs(cx - center_x) / width
    metrics['delta_x'] = delta_x
    metrics['centroid'] = (cx, cy)

    # 2. rᵥ - Void Ratio
    total_pixels = width * height
    material_pixels = len(material_coords)
    rv = 1 - (material_pixels / total_pixels)
    metrics['rv'] = rv

    # 3. ρᵣ - Packing Density
    min_x = np.min(x_coords)
    max_x = np.max(x_coords)
    min_y = np.min(y_coords)
    max_y = np.max(y_coords)

    bounding_area = (max_x - min_x + 1) * (max_y - min_y + 1)

    if bounding_area > 0:
        rho_r = material_pixels / bounding_area
    else:
        rho_r = 0

    metrics['rho_r'] = rho_r
    metrics['bounding_box'] = (min_x, max_x, min_y, max_y)

    # 4. μ - Cohesion
    distances = np.sqrt((x_coords - cx)**2 + (y_coords - cy)**2)
    std_dist = np.std(distances)
    max_dist = np.sqrt((width/2)**2 + (height/2)**2)

    mu = 1 - (std_dist / max_dist)
    mu = max(0, min(1, mu))  # Clamp to [0, 1]
    metrics['mu'] = mu

    # 5. xₚ - Peripheral Pull (Field Invariant)
    xp = (delta_x * 0.4) + (rv * 0.3) + ((1 - rho_r) * 0.2) + ((1 - mu) * 0.1)
    metrics['xp'] = xp

    # Store mask for visualization
    metrics['mask'] = material_mask

    return metrics


def interpret_metrics(metrics):
    """Provide human-readable interpretation of metrics."""
    interpretations = []

    delta_x = metrics['delta_x']
    rv = metrics['rv']
    rho_r = metrics['rho_r']
    mu = metrics['mu']
    xp = metrics['xp']

    # Δx interpretation
    if delta_x > 0.15:
        interpretations.append(f"⚠️ Δx = {delta_x:.3f}: Strong offset - compositional asymmetry")
    elif delta_x < 0.05:
        interpretations.append(f"🔴 Δx = {delta_x:.3f}: CENTERED - Radial Collapse Prior detected")
    else:
        interpretations.append(f"✅ Δx = {delta_x:.3f}: Moderate offset - balanced composition")

    # rᵥ interpretation
    if rv > 0.6:
        interpretations.append(f"⚠️ rᵥ = {rv:.3f}: High void - void-dominant composition")
    elif rv < 0.3:
        interpretations.append(f"🔴 rᵥ = {rv:.3f}: Low void - VOID COLLAPSE detected")
    else:
        interpretations.append(f"✅ rᵥ = {rv:.3f}: Balanced void ratio")

    # ρᵣ interpretation
    if rho_r > 0.7:
        interpretations.append(f"⚠️ ρᵣ = {rho_r:.3f}: High density - tight clustering/compression")
    elif rho_r < 0.3:
        interpretations.append(f"⚠️ ρᵣ = {rho_r:.3f}: Low density - dispersed/fragmented mass")
    else:
        interpretations.append(f"✅ ρᵣ = {rho_r:.3f}: Moderate packing density")

    # μ interpretation
    if mu > 0.7:
        interpretations.append(f"✅ μ = {mu:.3f}: High cohesion - stable, unified structure")
    elif mu < 0.4:
        interpretations.append(f"🔴 μ = {mu:.3f}: Low cohesion - FRAGMENTED structure")
    else:
        interpretations.append(f"⚠️ μ = {mu:.3f}: Moderate cohesion")

    # xₚ interpretation
    if xp > 0.4:
        interpretations.append(f"✅ xₚ = {xp:.3f}: Strong peripheral pull - ANTI-RCP behavior")
    elif xp < 0.2:
        interpretations.append(f"🔴 xₚ = {xp:.3f}: Weak peripheral pull - CENTRAL ATTRACTOR dominant")
    else:
        interpretations.append(f"⚠️ xₚ = {xp:.3f}: Moderate field behavior")

    return interpretations


def compare_methods(image_path):
    """Compare all three detection methods side by side."""
    methods = ['saliency', 'edges', 'otsu']
    results = {}

    for method in methods:
        print(f"\nTrying {method}...")
        metrics = calculate_kernel_metrics(image_path, method=method)
        if metrics:
            results[method] = metrics
            print(f"  ✓ Δx = {metrics['delta_x']:.3f}")
        else:
            print(f"  ✗ Failed")

    return results

def quick_diagnostic(metrics):
    """
    One-line compositional assessment.
    """
    dx = metrics['delta_x']
    xp = metrics['xp']
    rv = metrics['rv']

    if dx < 0.05 and xp < 0.3:
        diagnosis = "🔴 RADIAL COLLAPSE DETECTED - centered + weak peripheral pull"
    elif dx < 0.05 and xp > 0.4:
        diagnosis = "⚠️  CENTERED BUT ANTI-RCP - centered with strong peripheral features"
    elif dx > 0.15 and rv > 0.5:
        diagnosis = "✅ STRONG ASYMMETRY - off-center with good void discipline"
    elif dx > 0.10:
        diagnosis = "✅ MODERATE ASYMMETRY - compositional offset present"
    else:
        diagnosis = "⚠️  BALANCED - moderate across dimensions"

    print("\n" + "=" * 60)
    print("QUICK DIAGNOSTIC")
    print("=" * 60)
    print(diagnosis)
    print("=" * 60)

def print_method_statistics(all_results):
    """
    Print statistical comparison of methods.
    """
    if len(all_results) < 2:
        return

    print("\n" + "=" * 60)
    print("STATISTICAL COMPARISON")
    print("=" * 60)

    for metric in ['delta_x', 'rv', 'rho_r', 'mu', 'xp']:
        values = [m[metric] for m in all_results.values()]
        print(f"\n{metric}:")
        print(f"  Mean: {np.mean(values):.3f}")
        print(f"  Std:  {np.std(values):.3f}")
        print(f"  Min:  {np.min(values):.3f}")
        print(f"  Max:  {np.max(values):.3f}")

In [ ]:
# Calculate metrics using saliency detection (best for subject detection)
print("=" * 60)
print("ANALYZING WITH SALIENCY DETECTION")
print("=" * 60)

metrics = calculate_kernel_metrics(image_filename, method='saliency')

if metrics:
    print(f"\nDetection method: {metrics['method']}")
    print("\n" + "=" * 60)
    print("VTL KERNEL METRICS")
    print("=" * 60)
    print(f"\nΔx (Placement Offset):    {metrics['delta_x']:.3f}")
    print(f"rᵥ (Void Ratio):          {metrics['rv']:.3f}")
    print(f"ρᵣ (Packing Density):     {metrics['rho_r']:.3f}")
    print(f"μ (Cohesion):             {metrics['mu']:.3f}")
    print(f"xₚ (Peripheral Pull):     {metrics['xp']:.3f}")
    print(f"\nCentroid: ({metrics['centroid'][0]:.1f}, {metrics['centroid'][1]:.1f})")

    # Quick diagnostic
    quick_diagnostic(metrics)

    print("\n" + "=" * 60)
    print("DETAILED INTERPRETATION")
    print("=" * 60)

    interpretations = interpret_metrics(metrics)
    for interp in interpretations:
        print(f"\n{interp}")
else:
    print("Could not calculate metrics. No material detected in image.")

In [ ]:
# Compare all three methods to see which works best
print("\n" + "=" * 60)
print("COMPARING ALL DETECTION METHODS")
print("=" * 60)

all_results = compare_methods(image_filename)

# Show comparison table
if all_results:
    print("\n" + "=" * 60)
    print("METHOD COMPARISON")
    print("=" * 60)
    print(f"{'Method':<12} {'Δx':<8} {'rᵥ':<8} {'ρᵣ':<8} {'μ':<8} {'xₚ':<8}")
    print("-" * 60)

    for method, result in all_results.items():
        print(f"{method:<12} {result['delta_x']:<8.3f} {result['rv']:<8.3f} "
              f"{result['rho_r']:<8.3f} {result['mu']:<8.3f} {result['xp']:<8.3f}")

In [ ]:
def visualize_analysis(image_path, metrics):
    """Create visualization of the analysis."""
    # Load image
    img = Image.open(image_path).convert('RGB')
    img_array = np.array(img)

    height, width = img_array.shape[:2]

    # Get stored mask
    material_mask = metrics['mask']

    # Create figure
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # Original image with centroid and bounding box
    axes[0].imshow(img)

    # Plot centroid
    axes[0].plot(metrics['centroid'][0], metrics['centroid'][1],
                 'r+', markersize=20, markeredgewidth=3, label='Centroid')

    # Plot frame center
    axes[0].axvline(width/2, color='yellow', linestyle='--', alpha=0.7, linewidth=2, label='Frame Center')
    axes[0].axhline(height/2, color='yellow', linestyle='--', alpha=0.7, linewidth=2)

    # Plot bounding box
    bbox = metrics['bounding_box']
    rect = plt.Rectangle((bbox[0], bbox[2]), bbox[1]-bbox[0], bbox[3]-bbox[2],
                         fill=False, edgecolor='cyan', linewidth=2, label='Bounding Box')
    axes[0].add_patch(rect)

    axes[0].set_title('Original Image with Analysis', fontsize=14, fontweight='bold')
    axes[0].legend(loc='upper right')
    axes[0].axis('off')

    # Material mask
    axes[1].imshow(material_mask, cmap='RdYlBu_r')
    axes[1].plot(metrics['centroid'][0], metrics['centroid'][1],
                 'r+', markersize=15, markeredgewidth=2)
    axes[1].set_title('Material Mask (Detected)', fontsize=14, fontweight='bold')
    axes[1].axis('off')

    # Metrics visualization
    axes[2].axis('off')

    # Color code based on values
    dx_color = '🔴' if metrics['delta_x'] < 0.05 else '✅' if metrics['delta_x'] > 0.15 else '⚠️'
    rv_color = '🔴' if metrics['rv'] < 0.3 else '✅' if 0.3 <= metrics['rv'] <= 0.6 else '⚠️'
    rho_color = '✅' if 0.3 <= metrics['rho_r'] <= 0.7 else '⚠️'
    mu_color = '🔴' if metrics['mu'] < 0.4 else '✅' if metrics['mu'] > 0.7 else '⚠️'
    xp_color = '🔴' if metrics['xp'] < 0.2 else '✅' if metrics['xp'] > 0.4 else '⚠️'

    metrics_text = f"""KERNEL METRICS

{dx_color} Δx = {metrics['delta_x']:.3f}
   (Placement Offset)

{rv_color} rᵥ = {metrics['rv']:.3f}
   (Void Ratio)

{rho_color} ρᵣ = {metrics['rho_r']:.3f}
   (Packing Density)

{mu_color} μ = {metrics['mu']:.3f}
   (Cohesion)

{xp_color} xₚ = {metrics['xp']:.3f}
   (Peripheral Pull)

Centroid: ({metrics['centroid'][0]:.0f}, {metrics['centroid'][1]:.0f})
Frame Center: ({width/2:.0f}, {height/2:.0f})
Offset: {abs(metrics['centroid'][0] - width/2):.0f}px
"""
    axes[2].text(0.1, 0.5, metrics_text, fontsize=12, family='monospace',
                verticalalignment='center')
    axes[2].set_title('Measured Values', fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.show()

# Run visualization
visualize_analysis(image_filename, metrics)

In [ ]:
def visualize_all_methods(image_path, all_results):
    """
    Create side-by-side visualization of all three detection methods.
    """
    # Load image
    img = Image.open(image_path).convert('RGB')
    img_array = np.array(img)
    height, width = img_array.shape[:2]

    # Create figure with 3 rows, 3 columns
    fig, axes = plt.subplots(3, 3, figsize=(20, 18))

    methods = ['saliency', 'edges', 'otsu']
    method_names = ['Saliency Detection', 'Edge Detection', 'Otsu Thresholding']

    for idx, (method, method_name) in enumerate(zip(methods, method_names)):
        if method not in all_results:
            # Skip if method failed
            for col in range(3):
                axes[idx, col].axis('off')
                axes[idx, col].text(0.5, 0.5, f'{method_name}\nFailed',
                                   ha='center', va='center', fontsize=14)
            continue

        metrics = all_results[method]
        material_mask = metrics['mask']

        # Column 0: Original image with analysis
        axes[idx, 0].imshow(img)

        # Plot centroid
        axes[idx, 0].plot(metrics['centroid'][0], metrics['centroid'][1],
                         'r+', markersize=20, markeredgewidth=3, label='Centroid')

        # Plot frame center
        axes[idx, 0].axvline(width/2, color='yellow', linestyle='--',
                            alpha=0.7, linewidth=2, label='Frame Center')
        axes[idx, 0].axhline(height/2, color='yellow', linestyle='--',
                            alpha=0.7, linewidth=2)

        # Plot bounding box
        bbox = metrics['bounding_box']
        rect = plt.Rectangle((bbox[0], bbox[2]), bbox[1]-bbox[0], bbox[3]-bbox[2],
                             fill=False, edgecolor='cyan', linewidth=2, label='Bounding Box')
        axes[idx, 0].add_patch(rect)

        axes[idx, 0].set_title(f'{method_name}\nOriginal with Analysis',
                              fontsize=12, fontweight='bold')
        if idx == 0:
            axes[idx, 0].legend(loc='upper right', fontsize=8)
        axes[idx, 0].axis('off')

        # Column 1: Material mask
        axes[idx, 1].imshow(material_mask, cmap='RdYlBu_r')
        axes[idx, 1].plot(metrics['centroid'][0], metrics['centroid'][1],
                         'r+', markersize=15, markeredgewidth=2)
        axes[idx, 1].set_title(f'{method_name}\nMaterial Mask',
                              fontsize=12, fontweight='bold')
        axes[idx, 1].axis('off')

        # Column 2: Metrics
        axes[idx, 2].axis('off')

        # Color code based on values
        dx_color = '🔴' if metrics['delta_x'] < 0.05 else '✅' if metrics['delta_x'] > 0.15 else '⚠️'
        rv_color = '🔴' if metrics['rv'] < 0.3 else '✅' if 0.3 <= metrics['rv'] <= 0.6 else '⚠️'
        rho_color = '✅' if 0.3 <= metrics['rho_r'] <= 0.7 else '⚠️'
        mu_color = '🔴' if metrics['mu'] < 0.4 else '✅' if metrics['mu'] > 0.7 else '⚠️'
        xp_color = '🔴' if metrics['xp'] < 0.2 else '✅' if metrics['xp'] > 0.4 else '⚠️'

        metrics_text = f"""KERNEL METRICS

{dx_color} Δx = {metrics['delta_x']:.3f}
   Placement Offset

{rv_color} rᵥ = {metrics['rv']:.3f}
   Void Ratio

{rho_color} ρᵣ = {metrics['rho_r']:.3f}
   Packing Density

{mu_color} μ = {metrics['mu']:.3f}
   Cohesion

{xp_color} xₚ = {metrics['xp']:.3f}
   Peripheral Pull

Centroid:
({metrics['centroid'][0]:.0f}, {metrics['centroid'][1]:.0f})

Offset: {abs(metrics['centroid'][0] - width/2):.0f}px
"""
        axes[idx, 2].text(0.1, 0.5, metrics_text, fontsize=11, family='monospace',
                         verticalalignment='center')
        axes[idx, 2].set_title(f'{method_name}\nMeasured Values',
                              fontsize=12, fontweight='bold')

    plt.tight_layout()
    plt.show()

    # Print comparison table
    print("\n" + "=" * 80)
    print("METHOD COMPARISON TABLE")
    print("=" * 80)
    print(f"{'Method':<20} {'Δx':<10} {'rᵥ':<10} {'ρᵣ':<10} {'μ':<10} {'xₚ':<10}")
    print("-" * 80)

    for method in methods:
        if method in all_results:
            m = all_results[method]
            print(f"{method:<20} {m['delta_x']:<10.3f} {m['rv']:<10.3f} "
                  f"{m['rho_r']:<10.3f} {m['mu']:<10.3f} {m['xp']:<10.3f}")

    print("=" * 80)

    # Show which metrics agree
    if len(all_results) >= 2:
        print("\nAGREEMENT ANALYSIS:")
        print("-" * 80)

        delta_x_values = [m['delta_x'] for m in all_results.values()]
        delta_x_std = np.std(delta_x_values)

        if delta_x_std < 0.02:
            print("✅ All methods AGREE on placement (Δx variance < 0.02)")
        elif delta_x_std < 0.05:
            print("⚠️  Methods show MODERATE disagreement on placement")
        else:
            print("🔴 Methods show STRONG disagreement - manual review recommended")

        print(f"   Δx standard deviation: {delta_x_std:.3f}")
        print(f"   Δx range: {min(delta_x_values):.3f} - {max(delta_x_values):.3f}")


# Run the comparison
print("=" * 80)
print("COMPARING ALL THREE DETECTION METHODS")
print("=" * 80)

all_results = compare_methods(image_filename)

if all_results:
    visualize_all_methods(image_filename, all_results)
else:
    print("No methods succeeded in detecting material.")

if all_results and len(all_results) >= 2:
    print_method_statistics(all_results)

In [ ]:
def upload_and_process_multiple(method='saliency', show_method_choice=True):
    """
    Upload multiple images and process with chosen method.

    Parameters:
    - method: 'saliency', 'edges', or 'otsu' (default: 'saliency')
    - show_method_choice: print why this method was chosen
    """
    import pandas as pd

    print("=" * 60)
    print("BATCH IMAGE UPLOAD")
    print("=" * 60)

    if show_method_choice:
        print(f"Using detection method: {method.upper()}")
        if method == 'saliency':
            print("  → Best for: Images with gradient backgrounds, complex scenes")
        elif method == 'edges':
            print("  → Best for: Clean images with strong edges")
        elif method == 'otsu':
            print("  → Best for: High-contrast images with simple backgrounds")
        print()

    print("Select multiple images to upload...")

    uploaded_files = files.upload()

    if not uploaded_files:
        print("No files uploaded.")
        return None

    print(f"\n✓ Uploaded {len(uploaded_files)} files")

    results_table = []

    for filename in uploaded_files.keys():
        print(f"\n{'='*60}")
        print(f"Processing: {filename}")
        print('='*60)

        metrics = calculate_kernel_metrics(filename, method=method)

        if metrics:
            results_table.append({
                'image': filename,
                'delta_x': metrics['delta_x'],
                'rv': metrics['rv'],
                'rho_r': metrics['rho_r'],
                'mu': metrics['mu'],
                'xp': metrics['xp']
            })
            print(f"✓ Δx = {metrics['delta_x']:.3f}, xₚ = {metrics['xp']:.3f}")
        else:
            print("✗ Failed to process")

    df = pd.DataFrame(results_table)

    print("\n" + "=" * 80)
    print(f"BATCH RESULTS ({method.upper()})")
    print("=" * 80)
    print(df.to_string(index=False))

    if len(df) > 1:
        print("\n" + "=" * 80)
        print("STATISTICS")
        print("=" * 80)
        print(f"Mean Δx:  {df['delta_x'].mean():.3f} (std: {df['delta_x'].std():.3f})")
        print(f"Mean rᵥ:  {df['rv'].mean():.3f}")
        print(f"Mean xₚ:  {df['xp'].mean():.3f}")

    return df

# Use saliency (default)
batch_results = upload_and_process_multiple(method='saliency')

# Or try with edges:
# batch_results = upload_and_process_multiple(method='edges')

In [ ]:
def upload_and_process_multiple_all_methods():
    """
    Upload multiple images and compare all detection methods for each.
    Returns a DataFrame with results from all methods.
    """
    import pandas as pd

    print("=" * 60)
    print("BATCH IMAGE UPLOAD - ALL METHODS")
    print("=" * 60)
    print("Select multiple images to upload...")

    # Upload multiple files
    uploaded_files = files.upload()

    if not uploaded_files:
        print("No files uploaded.")
        return None

    print(f"\n✓ Uploaded {len(uploaded_files)} files")

    # Process each image with all three methods
    results_table = []

    for filename in uploaded_files.keys():
        print(f"\n{'='*60}")
        print(f"Processing: {filename}")
        print('='*60)

        # Try all three methods
        all_methods = compare_methods(filename)

        if all_methods:
            for method, metrics in all_methods.items():
                results_table.append({
                    'image': filename,
                    'method': method,
                    'delta_x': metrics['delta_x'],
                    'rv': metrics['rv'],
                    'rho_r': metrics['rho_r'],
                    'mu': metrics['mu'],
                    'xp': metrics['xp']
                })

            # Show quick comparison
            print(f"\nΔx comparison:")
            for method, metrics in all_methods.items():
                print(f"  {method:10} = {metrics['delta_x']:.3f}")
        else:
            print("✗ All methods failed")

    # Create DataFrame
    df = pd.DataFrame(results_table)

    # Print summary table
    print("\n" + "=" * 80)
    print("BATCH RESULTS - ALL METHODS")
    print("=" * 80)
    print(df.to_string(index=False))

    # Print method comparison statistics
    if len(df) > 0:
        print("\n" + "=" * 80)
        print("METHOD COMPARISON STATISTICS")
        print("=" * 80)

        for method in ['saliency', 'edges', 'otsu']:
            method_df = df[df['method'] == method]
            if len(method_df) > 0:
                print(f"\n{method.upper()}:")
                print(f"  Mean Δx:  {method_df['delta_x'].mean():.3f} (std: {method_df['delta_x'].std():.3f})")
                print(f"  Mean rᵥ:  {method_df['rv'].mean():.3f}")
                print(f"  Mean xₚ:  {method_df['xp'].mean():.3f}")

        # Count centered vs asymmetric across all images
        print("\n" + "=" * 80)
        print("COMPOSITIONAL PATTERNS (using saliency)")
        print("=" * 80)

        saliency_df = df[df['method'] == 'saliency']
        if len(saliency_df) > 0:
            centered = len(saliency_df[saliency_df['delta_x'] < 0.05])
            moderate = len(saliency_df[(saliency_df['delta_x'] >= 0.05) & (saliency_df['delta_x'] < 0.15)])
            asymmetric = len(saliency_df[saliency_df['delta_x'] >= 0.15])

            print(f"Centered (Δx < 0.05):      {centered}/{len(saliency_df)}")
            print(f"Moderate (0.05 ≤ Δx < 0.15): {moderate}/{len(saliency_df)}")
            print(f"Asymmetric (Δx ≥ 0.15):    {asymmetric}/{len(saliency_df)}")

    return df

# Run batch processing with all methods
batch_results = upload_and_process_multiple_all_methods()

In [ ]:
def export_results(metrics, image_filename, output_csv='kernel_metrics.csv'):
    """
    Export metrics to CSV for later analysis.
    """
    import csv
    import os

    # Prepare data
    row = {
        'image': image_filename,
        'delta_x': metrics['delta_x'],
        'rv': metrics['rv'],
        'rho_r': metrics['rho_r'],
        'mu': metrics['mu'],
        'xp': metrics['xp'],
        'centroid_x': metrics['centroid'][0],
        'centroid_y': metrics['centroid'][1],
        'method': metrics.get('method', 'unknown')
    }

    # Check if file exists
    file_exists = os.path.isfile(output_csv)

    # Write to CSV
    with open(output_csv, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=row.keys())

        if not file_exists:
            writer.writeheader()

        writer.writerow(row)

    print(f"✅ Results exported to {output_csv}")

    # Download the CSV
    files.download(output_csv)

# Usage:
# export_results(metrics, image_filename)

## **Limits of Kernel Metrics**

Kernel metrics measure *geometric composition*, not semantics. They quantify how visual mass is arranged within the frame, but they do not interpret meaning, identity, or correctness of depicted subjects. Several important limitations follow from this design.

### **1. Multi-Object Scenes**
The kernel assumes a dominant material cluster.  
Images with multiple equally weighted subjects can produce ambiguous or unstable centroid and cohesion measurements. In such cases, Δx and μ reflect aggregate geometry rather than discrete-object intent.

### **2. Ambiguous or Soft Boundaries**
Saliency, Otsu, and edge-based methods can disagree when subjects blend into backgrounds (fog, bokeh, smoke, soft gradients). This affects bounding boxes, packing density (ρᵣ), cohesion (μ), and void ratio (rᵥ).

Agreement analysis helps surface these cases, but ambiguity cannot always be eliminated.

### **3. High-Texture or Cluttered Backgrounds**
Where background texture is visually dominant—grass, foliage, cluttered interiors—detectors may over-select background material. This can inflate ρᵣ and distort rᵥ.

### **4. Extremely Minimal or Extremely Maximal Images**
Very sparse scenes (large voids) or nearly full-frame subjects (almost no void) can collapse metrics toward trivial extremes. These values are still geometrically valid but require contextual interpretation.

### **5. No Semantic Understanding**
Kernel metrics do **not** determine:

- prompt correctness  
- subject identity  
- object count  
- realism or narrative intent  

They measure **composition only**, not meaning.

### **6. Dependence on Subject Detection**
All metrics depend on an intermediate mask.  
If subject detection is inaccurate, Δx, rᵥ, ρᵣ, and μ inherit those errors. Offering three detection methods mitigates this, but mask selection remains the primary source of variance.

### **7. Horizontal Bias in Δx**
Δx measures horizontal displacement, which is the dominant axis of compositional prior in most generative models. Vertical displacement is not explicitly measured. This is intentional but should be noted as a constraint.

### **8. Not a Replacement for Full Evaluation Pipelines**
Kernel metrics complement, not replace:

- FID / KID (distribution fidelity)  
- CLIPScore (semantic alignment)  
- T2I-CompBench / GenEval (object correctness)  

They fill the **compositional geometry** gap, not semantic or realism gaps.

---

**These limits clarify that kernel metrics are a measurement instrument—not a perceptual model or aesthetic score. Their strength comes from focusing strictly on geometry while acknowledging where geometric measurement becomes ambiguous.**


Collapse ≠ failure. The center is the delta of choice. Any score can carry, as long as it has intent.

### **Mask Review**

Prompt you to upload an image
Generate 20 different masks including:

Edge Detection: Canny (2 variants), Sobel, Laplacian, Bilateral-enhanced
Thresholding: Otsu, Otsu inverse, Adaptive (Mean & Gaussian)
Saliency Maps: Spectral Residual, Fine Grained
Morphological: Gradient, Top Hat (bright features), Black Hat (dark features)
Structural: Contours, Distance Transform
Texture: Local Variance
Color Channels: HSV Value, HSV Saturation, LAB Luminance


Display everything in a clean grid for comparison

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
from PIL import Image
import io

def generate_all_masks(image_path):
    """
    Generate multiple types of masks from a single input image.
    """
    # Read image
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Dictionary to store all masks
    masks = {}

    # 1. CANNY EDGE DETECTION
    edges_canny = cv2.Canny(gray, 50, 150)
    masks['Canny Edges'] = edges_canny

    # 2. CANNY (Aggressive)
    edges_canny_agg = cv2.Canny(gray, 30, 100)
    masks['Canny Edges (Aggressive)'] = edges_canny_agg

    # 3. SOBEL EDGE DETECTION (X and Y)
    sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    sobel_combined = np.sqrt(sobelx**2 + sobely**2)
    sobel_combined = np.uint8(255 * sobel_combined / np.max(sobel_combined))
    masks['Sobel Edges'] = sobel_combined

    # 4. LAPLACIAN EDGE DETECTION
    laplacian = cv2.Laplacian(gray, cv2.CV_64F)
    laplacian = np.uint8(np.absolute(laplacian))
    masks['Laplacian'] = laplacian

    # 5. OTSU THRESHOLDING
    _, otsu = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    masks['Otsu Threshold'] = otsu

    # 6. OTSU INVERSE
    _, otsu_inv = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    masks['Otsu Inverse'] = otsu_inv

    # 7. ADAPTIVE THRESHOLD (Mean)
    adaptive_mean = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_MEAN_C,
                                          cv2.THRESH_BINARY, 11, 2)
    masks['Adaptive Threshold (Mean)'] = adaptive_mean

    # 8. ADAPTIVE THRESHOLD (Gaussian)
    adaptive_gaussian = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                              cv2.THRESH_BINARY, 11, 2)
    masks['Adaptive Threshold (Gaussian)'] = adaptive_gaussian

    # 9. MORPHOLOGICAL GRADIENT
    kernel = np.ones((3,3), np.uint8)
    gradient = cv2.morphologyEx(gray, cv2.MORPH_GRADIENT, kernel)
    masks['Morphological Gradient'] = gradient

    # 10. TOP HAT (bright features)
    tophat = cv2.morphologyEx(gray, cv2.MORPH_TOPHAT, kernel)
    masks['Top Hat (Bright Features)'] = tophat

    # 11. BLACK HAT (dark features)
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)
    masks['Black Hat (Dark Features)'] = blackhat

    # 12. SIMPLE SALIENCY (Spectral Residual)
    saliency = cv2.saliency.StaticSaliencySpectralResidual_create()
    success, saliency_map = saliency.computeSaliency(img)
    saliency_map = (saliency_map * 255).astype("uint8")
    masks['Saliency (Spectral Residual)'] = saliency_map

    # 13. FINE GRAINED SALIENCY
    saliency_fg = cv2.saliency.StaticSaliencyFineGrained_create()
    success, saliency_map_fg = saliency_fg.computeSaliency(img)
    saliency_map_fg = (saliency_map_fg * 255).astype("uint8")
    masks['Saliency (Fine Grained)'] = saliency_map_fg

    # 14. CONTOUR DETECTION
    contours, _ = cv2.findContours(edges_canny, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    contour_img = np.zeros_like(gray)
    cv2.drawContours(contour_img, contours, -1, (255), 1)
    masks['Contours'] = contour_img

    # 15. HIGH CONTRAST REGIONS (variance filter)
    kernel_size = 5
    mean_filtered = cv2.blur(gray, (kernel_size, kernel_size))
    variance = cv2.blur((gray - mean_filtered)**2, (kernel_size, kernel_size))
    variance_norm = np.uint8(255 * variance / np.max(variance))
    masks['Local Variance (Texture)'] = variance_norm

    # 16. DISTANCE TRANSFORM (from Otsu)
    dist_transform = cv2.distanceTransform(otsu, cv2.DIST_L2, 5)
    dist_transform = np.uint8(255 * dist_transform / np.max(dist_transform))
    masks['Distance Transform'] = dist_transform

    # 17. HSV - Value Channel
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    value_channel = hsv[:,:,2]
    masks['HSV Value Channel'] = value_channel

    # 18. HSV - Saturation Channel
    saturation_channel = hsv[:,:,1]
    masks['HSV Saturation Channel'] = saturation_channel

    # 19. LAB - Luminance
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    luminance = lab[:,:,0]
    masks['LAB Luminance'] = luminance

    # 20. BILATERAL EDGE ENHANCEMENT
    bilateral = cv2.bilateralFilter(gray, 9, 75, 75)
    enhanced_edges = cv2.Canny(bilateral, 50, 150)
    masks['Bilateral Enhanced Edges'] = enhanced_edges

    return img_rgb, masks

def display_masks(original_img, masks):
    """
    Display original image and all generated masks in a grid.
    """
    n_masks = len(masks) + 1  # +1 for original
    cols = 4
    rows = int(np.ceil(n_masks / cols))

    fig, axes = plt.subplots(rows, cols, figsize=(20, 5*rows))
    axes = axes.flatten()

    # Show original
    axes[0].imshow(original_img)
    axes[0].set_title('Original Image', fontsize=12, fontweight='bold')
    axes[0].axis('off')

    # Show all masks
    for idx, (name, mask) in enumerate(masks.items(), start=1):
        axes[idx].imshow(mask, cmap='gray')
        axes[idx].set_title(name, fontsize=10)
        axes[idx].axis('off')

    # Hide unused subplots
    for idx in range(n_masks, len(axes)):
        axes[idx].axis('off')

    plt.tight_layout()
    plt.show()

# MAIN EXECUTION
print("Upload your image file:")
uploaded = files.upload()

# Get the uploaded file
image_path = list(uploaded.keys())[0]

# Generate all masks
print("\nGenerating masks...")
original_img, masks = generate_all_masks(image_path)

print(f"Generated {len(masks)} different masks!")
print("\nAvailable masks:")
for i, name in enumerate(masks.keys(), 1):
    print(f"{i}. {name}")

# Display all results
display_masks(original_img, masks)

# Optional: Save individual masks
print("\nWould you like to save individual masks? (uncomment below)")
# for name, mask in masks.items():
#     filename = f"{name.replace(' ', '_').lower()}.png"
#     cv2.imwrite(filename, mask)
#     print(f"Saved: {filename}")